# Task 2, Parts C2-C4: Five Compressed Caches at One Matched INT4 Byte Budget

**Budget (per prompt):** actual serialized bytes of the uniform INT4 channel-wise-K / token-wise-V cache of the
pre-question context: M_4(L) = 2 n_l H_kv d_h [floor(L/64)(64*4/8 + 4) + 2 (L mod 64)], with FP16 scale + FP16 offset
per 64-element group and the trailing L mod 64 tokens in FP16.

| Policy | What is stored | Bytes counted |
|---|---|---|
| C2 `evict_sink_recent` | 8 sink + most recent FP16 positions | FP16 K/V + 2 range bounds |
| C2 `evict_question` | 8 sink + 64 recent + highest question-aware scores, FP16 | FP16 K/V + int16 index per kept position |
| C3 `quant_token_int4` | all positions; token-wise K and V, INT4 | M_4(L) (same groups/metadata) |
| C3 `quant_kivi_int4` | all positions; channel-wise K, token-wise V, INT4 (the reference) | M_4(L) |
| C4 `hybrid_fp16_int2` | 8 sink + 64 recent + question-ranked FP16; rest channel-K/token-V INT2 in complete 64-groups, remainder FP16 | FP16 + INT2 payload + metadata + int16 FP16 indices |

Retention is **shared across all layers and heads** (one global index set; every retained position stores all
28 x 2 KV heads). Question-aware scores: attention mass from the question-suffix queries (all heads/layers, FP32,
from captured query projections and the RoPE'd cached keys of the working SDPA model), average-pooled with kernel 7.
This is a simplified, fixed SnapKV-inspired policy (not the head-specific published algorithm); the 8/64 safeguards
and pooling were fixed a priori -- nothing is tuned on these prompts. Quantized caches are dequantized to FP16 before
SDPA; INT4 layouts also quantize suffix/generated tokens whenever they complete a 64-token group.

Each prompt: FP16 reference first (teacher-forced logits kept in memory for KL), then the five policies. Results are
appended per (prompt, policy), so the notebook resumes after an interruption.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task2_attention_kv")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task2", need_public=False, need_prefix_banks=False)

In [ ]:
import pandas as pd
import torch
from tqdm.auto import tqdm
from shared.src.io import read_jsonl, append_jsonl, save_json
from task2_attention_kv.src import kv_cache as KV

model, tok = KV.load_kv_model()
cases = [KV.prepare_case(tok, c) for c in read_jsonl(ctx.data / "kv_retrieval" / "kv_retrieval_cases.jsonl.gz")]
OUT = ctx.results / "kv"
OUT.mkdir(exist_ok=True)
per_prompt = OUT / "per_prompt.jsonl"
sel_path = OUT / "selections.jsonl"
done = {(r["id"], r["policy"]) for r in read_jsonl(per_prompt)} if per_prompt.exists() else set()
POLICIES = KV.make_policies()          # fp16 reference + 5 compressed
print([p.name for p in POLICIES], "done:", len(done))

In [ ]:
# warm-up so the first prompt's timings are not inflated
_ = KV.run_policy(model, tok, cases[0], KV.Policy())
for c in tqdm(cases):
    if all((c["id"], p.name) in done for p in POLICIES):
        continue
    ref = KV.run_policy(model, tok, c, POLICIES[0])
    ref_info = dict(tf_logits=ref["_tf_logits"], correct=ref["correct"])
    for p in POLICIES:
        if (c["id"], p.name) in done:
            continue
        r = ref if p.name == "fp16" else KV.run_policy(model, tok, c, p, ref=ref_info)
        r = dict(r)
        r.pop("_tf_logits")
        kept = r.pop("kept_fp16", None)
        if p.name == "fp16":
            r.update(kl_first=0.0, kl_tf_mean=0.0, ref_correct=ref["correct"])
        if p.name in ("evict_sink_recent", "evict_question", "hybrid_fp16_int2"):
            append_jsonl(sel_path, dict(id=c["id"], policy=p.name, kept_fp16=kept, evidence=c["evidence_tokens"], ctx_len=c["ctx_len"]))
        append_jsonl(per_prompt, r)
    torch.cuda.empty_cache()

In [ ]:
df = pd.DataFrame(read_jsonl(per_prompt))
assert df.groupby("policy").size().eq(len(cases)).all(), df.groupby("policy").size()
assert df[df.policy != "fp16"].within_budget.all(), "every compressed cache must fit the INT4 reference budget"
cols = ["correct", "answer_nll", "kl_first", "kl_tf_mean", "evidence_retained", "frac_fp16_bytes", "k_rel_err", "v_rel_err"]
df.groupby("policy")[cols].mean().loc[[p.name for p in POLICIES]]